In [22]:
from tensorflow.keras.applications import EfficientNetV2B0
from tensorflow.keras.models import Model
from tensorflow.keras.applications.efficientnet_v2 import preprocess_input 
from tensorflow.keras.layers import Dense,GlobalAveragePooling2D,Dropout
from sklearn.metrics import classification_report,confusion_matrix
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.optimizers import Adam
import numpy as np

In [23]:
training_path = "../Datasets/data/train/Medical/X-Ray"
testing_path = "../Datasets/data/test/Medical/X-Ray"
valid_path = "../Datasets/data/vaild/Medical/X-Ray"

In [24]:
train_dataug = ImageDataGenerator(
    preprocessing_function = preprocess_input,
    rotation_range = 10,
    zoom_range = 0.2,
    horizontal_flip = True
)
test_dataug = ImageDataGenerator(
    preprocessing_function = preprocess_input
)
valid_dataug = ImageDataGenerator(
    preprocessing_function = preprocess_input
)

In [25]:
training_set = train_dataug.flow_from_directory(
    training_path,
    target_size = (224,224),
    batch_size = 16,
    class_mode = "categorical"
)
valid_set = valid_dataug.flow_from_directory(
    valid_path,
    target_size = (224,224),
    batch_size = 16,
    class_mode = "categorical"
)
testing_set = test_dataug.flow_from_directory(
    testing_path,
    target_size = (224,224),
    batch_size = 16,
    class_mode = "categorical",
    shuffle = False
)

Found 33028 images belonging to 6 classes.
Found 4108 images belonging to 6 classes.
Found 7029 images belonging to 6 classes.


In [26]:
testing_set.class_indices

{'chest x-ray': 0,
 'elbow': 1,
 'forearm': 2,
 'hand': 3,
 'humerus': 4,
 'shoulder': 5}

In [27]:
base_model = EfficientNetV2B0(weights="imagenet",include_top=False,input_shape=(224,224,3))

In [28]:
base_model.trainable = False

In [29]:
x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(128,activation='relu')(x)
x = Dropout(0.5)(x)
x = Dense(256, activation="relu")(x)
x = Dropout(0.3)(x)
x = Dense(128, activation='relu')(x)
output = Dense(6,activation="sigmoid")(x)

In [30]:
model = Model(inputs = base_model.input,outputs = output)

In [31]:
model.compile(optimizer=Adam(learning_rate=0.0005),metrics=["accuracy"],loss="categorical_crossentropy")

In [32]:
early_stop = EarlyStopping(monitor="val_loss",patience=3,restore_best_weights=True)

In [33]:
class_weight = {0:1,1:1,2:2,3:2,4:3,5:1}

In [34]:
model.fit(training_set,validation_data=(valid_set),epochs=15,callbacks=[early_stop])

Epoch 1/15


I0000 00:00:1793159615.759879    1411 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_121493__.271


1985/2065 ━━━━━━━━━━━━━━━━━━━━ 16s 202ms/step - accuracy: 0.8529 - loss: 0.4232

I0000 00:00:1793160031.602154    1411 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_121493__.271


2065/2065 ━━━━━━━━━━━━━━━━━━━━ 497s 231ms/step - accuracy: 0.9159 - loss: 0.2480 - val_accuracy: 0.9647 - val_loss: 0.1077
Epoch 2/15
2065/2065 ━━━━━━━━━━━━━━━━━━━━ 397s 192ms/step - accuracy: 0.9539 - loss: 0.1404 - val_accuracy: 0.9649 - val_loss: 0.1008
Epoch 3/15
2065/2065 ━━━━━━━━━━━━━━━━━━━━ 477s 231ms/step - accuracy: 0.9613 - loss: 0.1177 - val_accuracy: 0.9710 - val_loss: 0.0932
Epoch 4/15
2065/2065 ━━━━━━━━━━━━━━━━━━━━ 410s 198ms/step - accuracy: 0.9660 - loss: 0.1073 - val_accuracy: 0.9647 - val_loss: 0.1093
Epoch 5/15
2065/2065 ━━━━━━━━━━━━━━━━━━━━ 334s 162ms/step - accuracy: 0.9680 - loss: 0.1015 - val_accuracy: 0.9727 - val_loss: 0.0872
Epoch 6/15
2065/2065 ━━━━━━━━━━━━━━━━━━━━ 316s 153ms/step - accuracy: 0.9700 - loss: 0.0960 - val_accuracy: 0.9747 - val_loss: 0.0862
Epoch 7/15
2065/2065 ━━━━━━━━━━━━━━━━━━━━ 321s 155ms/step - accuracy: 0.9699 - loss: 0.0942 - val_accuracy: 0.9737 - val_loss: 0.0810
Epoch 8/15
2065/2065 ━━━━━━━━━━━━━━━━━━━━ 344s 167ms/step - accuracy: 0.9

In [43]:
prediction = model.predict(testing_set)

440/440 ━━━━━━━━━━━━━━━━━━━━ 40s 86ms/step


In [44]:
true_prediction = np.argmax(prediction,axis=1)

In [38]:
y_true = testing_set.classes

In [45]:
print(classification_report(y_true,true_prediction))
confusion_matrix(y_true,true_prediction)


              precision    recall  f1-score   support

           0       1.00      1.00      1.00      1550
           1       0.94      0.99      0.97      1951
           2       0.96      0.89      0.92       767
           3       0.99      1.00      0.99       879
           4       0.96      0.79      0.87       538
           5       0.96      0.99      0.97      1344

    accuracy                           0.97      7029
   macro avg       0.97      0.94      0.95      7029
weighted avg       0.97      0.97      0.97      7029



array([[1550,    0,    0,    0,    0,    0],
       [   0, 1930,   11,    1,    7,    2],
       [   0,   72,  683,    7,    3,    2],
       [   0,    1,    0,  878,    0,    0],
       [   0,   41,   18,    1,  425,   53],
       [   1,    3,    0,    1,    7, 1332]])

In [46]:
model.save("X_ray_category_model.keras")

In [41]:
# 'chest x-ray': 0,
#  'elbow': 1,
#  'forearm': 2,
#  'hand': 3,
#  'humerus': 4,
#  'shoulder': 5

In [42]:
#               precision    recall  f1-score   support

#            0       1.00      1.00      1.00      1550
#            1       0.91      0.99      0.95      1951
#            2       0.97      0.81      0.88       767
#            3       1.00      1.00      1.00       879
#            4       0.96      0.82      0.89       538
#            5       0.97      0.99      0.98      1542

#     accuracy                           0.96      7227
#    macro avg       0.97      0.93      0.95      7227
# weighted avg       0.96      0.96      0.96      7227

# array([[1550,    0,    0,    0,    0,    0],
#        [   0, 1936,    7,    0,    5,    3],
#        [   0,  132,  620,    3,    5,    7],
#        [   0,    2,    0,  876,    0,    1],
#        [   0,   43,   12,    1,  441,   41],
#        [   5,    3,    0,    0,    6, 1528]])